# Speculative Decoding - 실습 코드 1: Speculative Decoding (vLLM)

- Tutorial ID: `expand-speculative-decoding`
- Tutorial: Speculative Decoding
- Section ID: `expand-speculative-decoding-code-1`
- Section: 실습 코드 1: Speculative Decoding (vLLM)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: Speculative Decoding (vLLM)
#
# 이 노트북은 "개념 설명 → 작은 시뮬레이션으로 감 잡기 → 실제 vLLM 코드" 순서로
# 구성되어 있습니다. 특히 vLLM 부분은 실제 GPU와 대형 모델(Llama-3-70B/8B)이
# 필요하므로, 코드를 눈으로 읽고 "각 인자가 무엇을 뜻하는지" 이해하는 데
# 집중해도 충분합니다.
#
# 학습 목표:
#   1) Speculative Decoding이 왜, 어떻게 LLM 생성 속도를 높이는지 원리를 이해합니다.
#   2) draft(작은) 모델과 target(큰) 모델이 서로 어떤 역할을 하는지 파악합니다.
#   3) "수락(accept)/거절(reject)" 규칙이 왜 결과 품질을 그대로 유지하는지
#      (lossless) 직접 숫자로 확인합니다.
#   4) vLLM 코드에서 num_speculative_tokens, temperature 같은 옵션이 실제로
#      무엇을 조절하는지 이해합니다.
#
# 읽는 순서:
#   1) [개념] Speculative Decoding이 해결하려는 문제(느린 autoregressive 생성)를
#      먼저 읽습니다.
#   2) [개념] draft 모델과 target 모델의 역할, accept/reject 규칙을 글로 따라갑니다.
#   3) [미니 실습] 작은 확률분포로 accept/reject를 직접 코드로 실행해보고,
#      수락률을 눈으로 확인합니다.
#   4) [실전 코드] 실제 vLLM 코드에서 각 줄이 위 개념의 어떤 부분에 대응하는지
#      주석을 따라갑니다.
#   5) [실험 아이디어] num_speculative_tokens, temperature 등을 바꾸면 무엇이
#      달라질지 생각해봅니다.
#
# 주의:
#   - 미니 시뮬레이션(3번)은 순수 Python(random 모듈)만 사용하므로 Colab/로컬
#     어디서든 바로 실행됩니다. 단, 위에서부터 순서대로 실행해야 합니다
#     (뒤 셀이 앞 셀에서 정의한 함수/변수를 그대로 이어받아 사용합니다).
#   - 실전 코드(4번)는 torch/vLLM 및 GPU가 필요하므로 Colab(GPU 런타임)이나
#     자체 GPU 서버 실행을 권장합니다.
#   - vLLM은 업데이트가 매우 빠른 라이브러리라, 인자 이름이 버전에 따라 자주
#     바뀝니다. 실행 전 vLLM 공식 문서(https://docs.vllm.ai)에서 최신 사용법을
#     한 번 확인하는 습관을 들이세요.
# ============================================================


## 1. 문제 정의 — LLM 생성은 왜 느릴까?

### 1-1. Autoregressive(자기회귀적) 생성의 한계

LLM은 다음 토큰을 예측할 때 **이전에 만든 토큰들을 전부 다시 참고해서 딱 1개의 새로운 토큰만** 만들어냅니다.
그리고 그 토큰이 다음 예측의 입력이 되므로, 토큰을 100개 만들려면 **모델 전체를 100번 순차적으로 실행**해야 합니다.

```
토큰1 생성 → 토큰2 생성 → 토큰3 생성 → ... → 토큰100 생성
(화살표 하나하나가 모델 전체를 한 번 통과하는 forward pass입니다)
```

문제는 이 forward pass 한 번 한 번이 절대 가볍지 않다는 점입니다. 70B(700억) 파라미터 모델이라면,
토큰 1개를 만들기 위해 700억 개의 가중치를 대부분 GPU 메모리에서 읽어와야 합니다.

### 1-2. 진짜 병목은 "계산"이 아니라 "가중치를 읽어오는 시간"

직관적으로는 "모델이 크니까 계산이 오래 걸려서 느리다"고 생각하기 쉽지만, 실제로는 조금 다릅니다.

> **비유**: 도서관 서고에 가서 두꺼운 책 한 권을 통째로 꺼내와서, 딱 한 줄만 베껴 쓰고, 다시 서고에 반납하는 상황을 상상해보세요.
> 이 과정을 문장 100개를 베낄 때까지 100번 반복한다면 — 정작 오래 걸리는 건 "한 줄 베끼는 시간"이 아니라
> "서고까지 왔다갔다하며 책을 꺼내고 반납하는 시간"입니다.

LLM도 마찬가지입니다. 토큰 1개를 계산하는 데 필요한 실제 연산(행렬곱) 자체는 최신 GPU에게 매우 빠르게 끝나지만,
**그 연산에 쓸 가중치를 GPU 메모리(HBM)에서 연산장치로 옮겨오는 시간**이 오히려 더 오래 걸립니다.
이런 상황을 "memory-bandwidth-bound(메모리 대역폭이 속도를 결정함)"라고 부릅니다.

**핵심 인사이트**: 만약 가중치를 한 번 읽어올 때(= forward pass 한 번)마다 토큰을 1개가 아니라
5개, 10개씩 한꺼번에 "검증"할 수 있다면 어떨까요? 책을 한 번 꺼내올 때 다섯 문장을 베낄 수 있다면,
왕복 횟수가 줄어드는 만큼 전체 시간도 줄어들 것입니다.

Speculative Decoding은 바로 이 아이디어를 이용합니다.


## 2. 핵심 아이디어 — "빠른 초안 + 한 번에 검증"

### 2-1. 두 개의 모델을 사용합니다

| 모델 | 비유 | 특징 |
|---|---|---|
| Draft Model (초안 모델) | 빠른 신입 작가 | 크기가 작고 빠르지만, 정확도는 target보다 떨어짐 |
| Target Model (정답 모델) | 꼼꼼한 편집장 | 크기가 크고 느리지만, 우리가 최종적으로 원하는 "진짜" 품질 |

원본 코드 기준으로는 `meta-llama/Meta-Llama-3-8B-Instruct`(80억 파라미터)가 draft,
`meta-llama/Meta-Llama-3-70B-Instruct`(700억 파라미터)가 target입니다.

### 2-2. 진행 순서 (한 라운드 기준)

1. **[Draft가 제안]** 작은 draft 모델이 다음 토큰 후보를 K개(예: 5개) 빠르게 연속 생성합니다.
   draft 모델은 작으므로 5번을 순차 실행해도 여전히 빠릅니다.
2. **[Target이 한 번에 검증]** 큰 target 모델이 "이 5개 토큰이 이어졌을 때"를 **딱 한 번의 forward pass**로 채점합니다.
   마치 선생님이 학생이 낸 답 5개를 한 번에 훑어보며 채점하는 것과 비슷합니다.
   (training 때 쓰는 teacher-forcing과 원리가 같아서, 5개 위치의 확률을 병렬로 한 번에 계산할 수 있습니다.)
3. **[수락/거절 판정]** 앞에서부터 순서대로, 각 토큰을 다음 규칙으로 수락하거나 거절합니다.
   - target이 그 토큰을 draft보다 **더** 선호한다면 → **무조건 수락**
   - target이 그 토큰을 draft보다 **덜** 선호한다면 → **그 선호 비율만큼만 확률적으로 수락**, 나머지 확률로는 거절
   - 처음 거절이 발생한 지점부터는 뒤의 제안들을 모두 버리고, target 모델이 직접 "보정된 확률분포"에서 새 토큰 하나를 뽑아 대체합니다.
   - 만약 K개가 전부 수락되면? target은 이미 다음 위치의 확률분포도 같은 forward pass에서 공짜로 계산해뒀으므로, 보너스 토큰 1개를 추가로 뽑아줍니다.

### 2-3. 왜 결과 품질이 떨어지지 않을까? (Lossless 보장)

가장 신기한 부분은, 이 accept/reject 규칙을 수학적으로 잘 설계하면
**최종적으로 뽑히는 토큰의 확률분포가 target 모델 혼자 생성했을 때와 정확히 동일**하다는 점입니다.
즉, draft 모델이 다소 부정확해도 그 부정확함은 결과물에 전혀 영향을 주지 않고, **오직 속도에만 영향**을 줍니다.
(왜 이런 성질이 성립하는지는 아래 3번 미니 시뮬레이션에서 직접 숫자로 확인해봅니다.)


## 3. 미니 시뮬레이션 — 숫자로 accept/reject 직접 확인하기

실제 LLM의 어휘(vocabulary)는 수만~수십만 개 토큰이지만, 원리를 이해하기 위해
아주 작은 예시(후보 토큰 4개)로 accept/reject 로직을 직접 코드로 실행해보겠습니다.

**상황 설정**: "오늘 날씨가 좋습니 ___" 다음에 올 토큰을 예측한다고 가정합니다.
후보는 `"다"`, `"요"`, `"네"`, `"죠"` 4가지뿐이라고 단순화하겠습니다.

- `p_target`: 700억 파라미터 target 모델이 "진짜" 정확하게 계산했다고 가정한 확률분포 (우리가 원하는 정답 분포)
- `q_draft`: 80억 파라미터 draft 모델이 계산했다고 가정한 확률분포 (target과 비슷하지만 살짝 다름)

이 셀부터는 순수 Python(`random` 모듈)만 사용하므로 별도 설치 없이 바로 실행할 수 있습니다.
단, **아래 셀들은 위에서부터 순서대로 실행**해주세요 (뒤 셀이 앞 셀의 함수/변수를 그대로 이어받아 사용합니다).


In [ ]:
import random

random.seed(42)  # 실행할 때마다 같은 결과가 나오도록 고정 (재현성)

# -----------------------------------------------------------
# 아주 작은 예시 어휘 (실제 LLM은 토큰이 수만~수십만 개지만, 여기선 4개로 단순화)
# "오늘 날씨가 좋습니 ___" 다음에 올 토큰 후보라고 생각해주세요.
# -----------------------------------------------------------
vocab = ["다", "요", "네", "죠"]

# target 모델(700억 파라미터, 정확하지만 느림)이 계산했다고 가정한 "진짜" 확률분포
# 실제로는 이 값을 얻으려면 70B 모델 forward pass가 필요하지만, 여기서는 숫자로 흉내만 냅니다.
p_target = {"다": 0.50, "요": 0.30, "네": 0.15, "죠": 0.05}

# draft 모델(80억 파라미터, 빠르지만 약간 부정확)이 계산했다고 가정한 확률분포
# target과 "비슷하지만 완벽히 같지는 않게" 일부러 차이를 줬습니다.
q_draft = {"다": 0.35, "요": 0.40, "네": 0.10, "죠": 0.15}

# 두 확률분포가 각각 합이 1인지 확인 (확률분포의 기본 조건)
print("p_target 합:", sum(p_target.values()))
print("q_draft  합:", sum(q_draft.values()))


def sample_from(dist: dict) -> str:
    # 확률분포(dict: {토큰: 확률})에서 토큰을 하나 뽑는 함수.
    # 예) sample_from({"다": 0.6, "요": 0.4}) -> 60% 확률로 "다", 40% 확률로 "요"를 반환
    tokens = list(dist.keys())
    weights = list(dist.values())
    return random.choices(tokens, weights=weights, k=1)[0]


p_target 합: 1.0
q_draft  합: 1.0


### 3-1. accept/reject 규칙을 코드로 구현하기

앞에서 설명한 규칙을 그대로 코드로 옮기면 다음과 같습니다.

- **수락 확률**: `min(1, p_target(x) / q_draft(x))`
  - target이 draft보다 그 토큰을 더 좋아하면(p ≥ q) → 비율이 1 이상이 되어 **100% 수락**
  - target이 draft보다 그 토큰을 덜 좋아하면(p < q) → 그 비율만큼만 수락
- **거절되면**, "target이 draft보다 더 좋아했던 부분만 남긴" 새 확률분포에서 다시 뽑습니다.
  - 수식: `residual(x) = max(0, p_target(x) - q_draft(x))`, 그 다음 합이 1이 되도록 재정규화(normalize)


In [ ]:
def speculative_step(p_target: dict, q_draft: dict):
    # Speculative Decoding의 핵심 로직: 토큰 1개에 대한 accept/reject를 한 번 실행합니다.
    # 반환값:
    #   token       : 최종적으로 확정된 토큰
    #   accepted    : draft의 제안이 그대로 수락되었는지 (True/False)
    #   accept_prob : 이번에 계산된 수락 확률 (기록/출력용)

    # 1) draft 모델이 먼저 후보 토큰 x를 하나 제안합니다.
    x = sample_from(q_draft)

    p_x = p_target[x]  # target이 이 토큰을 얼마나 좋아하는지
    q_x = q_draft[x]   # draft가 이 토큰을 얼마나 좋아하는지 (이미 뽑았으므로 q_x > 0)

    # 2) 수락 확률 계산: target이 draft보다 이 토큰을 더/덜 좋아하는 정도의 비율
    accept_prob = min(1.0, p_x / q_x)

    # 3) 동전을 던지듯 0~1 사이 난수를 뽑아서 수락 확률과 비교
    coin = random.random()

    if coin < accept_prob:
        # target도 이 토큰을 충분히 좋아함 -> draft의 제안을 그대로 수락
        return x, True, accept_prob
    else:
        # 거절: target이 draft보다 "더 선호했던 부분"만 남겨서 재정규화한 분포를 새로 만듭니다.
        residual = {t: max(0.0, p_target[t] - q_draft[t]) for t in p_target}
        total = sum(residual.values())
        residual = {t: v / total for t, v in residual.items()}

        # 이 보정된 분포에서 target 모델을 대신해 새 토큰을 뽑습니다.
        new_token = sample_from(residual)
        return new_token, False, accept_prob


# 한 번 실행해서 결과를 살펴봅시다.
token, accepted, accept_prob = speculative_step(p_target, q_draft)
print(f"제안된/확정된 토큰: '{token}'")
print(f"수락 여부: {accepted}")
print(f"이번 토큰의 수락 확률: {accept_prob:.2%}")


제안된/확정된 토큰: '요'
수락 여부: True
이번 토큰의 수락 확률: 75.00%


### 3-2. 수천 번 반복해서 "정말로 target과 같은 분포가 나오는지" 확인하기

한 번의 결과만 보면 우연인지 아닌지 알 수 없습니다. 20,000번 반복해서
1) 실제 평균 수락률이 이론값과 비슷한지, 2) 최종적으로 뽑히는 토큰의 비율이 `p_target`과 일치하는지 확인해보겠습니다.

이론적으로 기대되는 수락률은 각 토큰에 대해 `min(p_target(x), q_draft(x))`를 모두 더한 값입니다.
(draft가 그 토큰을 제안할 확률과, 그 토큰이 수락될 확률을 곱해서 전체 토큰에 대해 더한 값과 같습니다.)


In [ ]:
# 이론적으로 기대되는 수락률 = Σ min(p_target(x), q_draft(x))
theoretical_accept_rate = sum(min(p_target[t], q_draft[t]) for t in vocab)
print(f"이론적으로 기대되는 수락률: {theoretical_accept_rate:.1%}\n")

trials = 20000
accept_count = 0
final_token_counts = {t: 0 for t in vocab}

for _ in range(trials):
    token, accepted, _ = speculative_step(p_target, q_draft)
    if accepted:
        accept_count += 1
    final_token_counts[token] += 1

print(f"[{trials}번 시뮬레이션 결과]")
print(f"실제 수락률: {accept_count / trials:.1%}  (이론값 {theoretical_accept_rate:.1%}과 비교)\n")

print("최종적으로 확정된 토큰의 비율 vs target 모델의 '진짜' 확률 (p_target):")
for t in vocab:
    sim_ratio = final_token_counts[t] / trials
    print(f"  '{t}': 시뮬레이션 {sim_ratio:.1%}   |   p_target {p_target[t]:.1%}")


이론적으로 기대되는 수락률: 80.0%

[20000번 시뮬레이션 결과]
실제 수락률: 79.7%  (이론값 80.0%과 비교)

최종적으로 확정된 토큰의 비율 vs target 모델의 '진짜' 확률 (p_target):
  '다': 시뮬레이션 50.3%   |   p_target 50.0%
  '요': 시뮬레이션 30.1%   |   p_target 30.0%
  '네': 시뮬레이션 14.8%   |   p_target 15.0%
  '죠': 시뮬레이션 4.9%   |   p_target 5.0%


> 💡 **여기서 확인한 것**: draft 모델(`q_draft`)은 실제 target 모델(`p_target`)과 분명히 다른, "부정확한" 확률분포를 가지고 있었습니다.
> 그런데도 accept/reject 규칙을 통과한 **최종 결과물의 분포는 `p_target`과 사실상 동일**합니다.
> 즉, draft 모델이 얼마나 정확한지는 **속도(수락률)**에만 영향을 주고, **최종 결과의 품질에는 영향을 주지 않습니다.**
> 이것이 Speculative Decoding이 "품질 저하 없는(lossless) 가속 기법"이라고 불리는 이유입니다.

다음 섹션에서는 토큰 1개가 아니라 `num_speculative_tokens`개(예: 5개)를 한 번에 제안하고 검증하는,
**실제 vLLM과 동일한 구조**로 시뮬레이션을 확장해보겠습니다.


## 4. 한 라운드에 여러 토큰 제안하기 (num_speculative_tokens 흉내내기)

실제 vLLM 코드에서는 `num_speculative_tokens=5`처럼, draft 모델이 **한 번에 여러 개(K개)의 토큰을 연속으로 제안**합니다.
target 모델은 이 K개를 한 번의 forward pass로 검증하고, **앞에서부터 순서대로** accept/reject를 적용하다가
처음 거절이 발생하는 순간 멈추고 그 자리를 보정합니다. (뒤에 남은 제안들은 버려집니다.)

- 최선의 경우: K개 모두 수락 + 보너스 토큰 1개 → forward pass 1번으로 토큰 K+1개 획득
- 최악의 경우: 첫 제안부터 거절 → forward pass 1번으로 토큰 1개만 획득 (그래도 손해는 아닙니다 — 어차피 1개는 만들어야 했으니까요)

(실제로는 각 위치(position)마다 문맥이 달라지므로 확률분포도 함께 달라지지만,
여기서는 이해를 돕기 위해 매 단계 같은 `p_target`, `q_draft`를 재사용합니다.)


In [ ]:
def speculative_decode_round(p_target: dict, q_draft: dict, num_speculative_tokens: int = 5, verbose: bool = True):
    # vLLM의 num_speculative_tokens 파라미터에 대응하는 '한 라운드'를 시뮬레이션합니다.
    # draft가 최대 num_speculative_tokens개를 제안하고, target이 한 번에 검증합니다.
    accepted_tokens = []
    all_accepted = True

    for i in range(num_speculative_tokens):
        token, accepted, accept_prob = speculative_step(p_target, q_draft)
        accepted_tokens.append(token)

        if accepted:
            if verbose:
                print(f"  {i+1}번째 제안 '{token}' → 수락 (수락확률 {accept_prob:.0%}) ✅")
        else:
            if verbose:
                print(f"  {i+1}번째 제안 → 거절, target이 '{token}'(으)로 교정 후 이번 라운드 종료 ❌")
            all_accepted = False
            break

    if all_accepted:
        # for문이 중간에 멈추지 않고 K개 모두 수락되었다면,
        # target의 다음 위치 확률분포에서 보너스 토큰 1개를 추가로 뽑습니다.
        bonus = sample_from(p_target)
        accepted_tokens.append(bonus)
        if verbose:
            print(f"  🎉 {num_speculative_tokens}개 전부 수락! 보너스 토큰 '{bonus}' 추가 생성")

    return accepted_tokens


random.seed(0)
print("=== num_speculative_tokens=5 로 3라운드 시뮬레이션 ===\n")
for round_num in range(1, 4):
    print(f"[라운드 {round_num}]")
    tokens = speculative_decode_round(p_target, q_draft, num_speculative_tokens=5)
    print(f"  → 이번 라운드 확정 토큰: {tokens}  (target forward pass는 딱 1번만 호출됨)\n")


=== num_speculative_tokens=5 로 3라운드 시뮬레이션 ===

[라운드 1]
  1번째 제안 '네' → 수락 (수락확률 100%) ✅
  2번째 제안 '요' → 수락 (수락확률 75%) ✅
  3번째 제안 '요' → 수락 (수락확률 75%) ✅
  4번째 제안 '네' → 수락 (수락확률 100%) ✅
  5번째 제안 '요' → 수락 (수락확률 75%) ✅
  🎉 5개 전부 수락! 보너스 토큰 '네' 추가 생성
  → 이번 라운드 확정 토큰: ['네', '요', '요', '네', '요', '네']  (target forward pass는 딱 1번만 호출됨)

[라운드 2]
  1번째 제안 '요' → 수락 (수락확률 75%) ✅
  2번째 제안 '네' → 수락 (수락확률 100%) ✅
  3번째 제안 '다' → 수락 (수락확률 100%) ✅
  4번째 제안 → 거절, target이 '네'(으)로 교정 후 이번 라운드 종료 ❌
  → 이번 라운드 확정 토큰: ['요', '네', '다', '네']  (target forward pass는 딱 1번만 호출됨)

[라운드 3]
  1번째 제안 '다' → 수락 (수락확률 100%) ✅
  2번째 제안 → 거절, target이 '다'(으)로 교정 후 이번 라운드 종료 ❌
  → 이번 라운드 확정 토큰: ['다', '다']  (target forward pass는 딱 1번만 호출됨)



## 5. 실전 코드 — vLLM으로 Speculative Decoding 실행하기

이제 위에서 이해한 개념이 실제 vLLM 코드의 어느 부분에 대응하는지 하나씩 짚어보겠습니다.

⚠️ **실행 전 확인 사항**

- 이 코드는 **GPU가 필요**하며, target 모델(70B)과 draft 모델(8B)을 합쳐 수백 GB급 GPU 메모리를 사용합니다.
  개인 실습 환경(Colab 무료 GPU 등)에서는 그대로 실행하기 어려울 수 있습니다. (더 가벼운 모델 조합은 섹션 6-2 참고)
- vLLM은 매우 빠르게 업데이트되는 라이브러리라 **인자 이름이 버전마다 바뀔 수 있습니다.**
  실제로 이 노트북을 준비하며 vLLM 공식 문서를 확인해보니, 예전에 쓰이던
  `speculative_model=`, `num_speculative_tokens=`처럼 각각 따로 넘기던 방식은 이미 폐기(deprecated)되었고,
  지금은 아래처럼 `speculative_config={...}` 딕셔너리 하나로 모아서 넘기는 방식이 공식 사용법입니다.
  아래 코드는 이 최신 방식 기준으로 작성했지만, 실행 전에는 [vLLM 공식 문서](https://docs.vllm.ai)에서
  현재 설치된 버전 기준 최신 사용법을 한 번 더 확인하는 것을 권장합니다.


In [ ]:
# vLLM이 설치되어 있지 않다면 아래 주석을 풀고 먼저 설치하세요. (GPU 환경 필요)
# !pip install vllm

from vllm import LLM, SamplingParams

# -----------------------------------------------------------
# LLM 객체 생성: target 모델을 불러오면서, speculative_config로
# draft 모델과 speculative decoding 관련 설정을 함께 지정합니다.
# -----------------------------------------------------------
llm = LLM(
    # model: "target" 모델 - 우리가 최종적으로 원하는 품질의, 크고 정확한 모델
    #        (위 개념 설명의 p_target 역할)
    model="meta-llama/Meta-Llama-3-70B-Instruct",

    # speculative_config: speculative decoding과 관련된 설정을 모아두는 딕셔너리입니다.
    speculative_config={
        # method: 초안(draft) 토큰을 어떤 방식으로 만들지 지정합니다.
        #         "draft_model"은 우리가 배운 것처럼 "별도의 작은 모델"을 쓰는
        #         가장 고전적인 방식입니다.
        #         (다른 선택지: "eagle", "eagle3", "mtp", "ngram", "medusa", "suffix" 등 -> 섹션 6-4 참고)
        "method": "draft_model",

        # model: "draft" 모델 - 빠르게 후보 토큰을 제안하는 작고 가벼운 모델
        #        (위 개념 설명의 q_draft 역할)
        "model": "meta-llama/Meta-Llama-3-8B-Instruct",

        # num_speculative_tokens: draft 모델이 한 라운드에 제안할 토큰 개수 (K)
        #        위 섹션 4 시뮬레이션의 num_speculative_tokens와 동일한 역할입니다.
        #        너무 크면(예: 20) 뒤쪽 제안이 거절될 확률이 높아져 낭비가 커지고,
        #        너무 작으면(예: 1) 한 번에 벌어들이는 토큰 수가 줄어 속도 이득이 줄어듭니다.
        #        보통 3~7 사이 값으로 실험하며 최적값을 찾습니다.
        "num_speculative_tokens": 5,
    },
)

# -----------------------------------------------------------
# SamplingParams: "어떻게 토큰을 뽑을지" 정의 (target 모델 기준)
# -----------------------------------------------------------
sampling_params = SamplingParams(
    # temperature: 확률분포를 얼마나 "뾰족하게" 또는 "평평하게" 만들지 조절하는 값
    #   - temperature=0.0 -> 확률이 가장 높은 토큰만 고르는 greedy decoding (deterministic)
    #   - temperature가 높을수록 -> 다양한 토큰이 골고루 뽑힐 확률이 커짐 (더 "무작위")
    #
    #   왜 temperature=0에서 수락률이 가장 높은 경향이 있을까요?
    #   temperature=0이면 draft/target 둘 다 "가장 확률 높은 토큰 1개"만 사실상 고르게
    #   되므로, 두 모델이 같은 문제(다음 토큰 맞히기)에서 종종 같은 답을 고르기 쉬워지고,
    #   결과적으로 accept 조건(target이 draft의 제안을 충분히 선호)이 더 자주
    #   만족되는 경향이 있기 때문입니다.
    temperature=0.0,

    # max_tokens: 최대 몇 개의 토큰까지 생성할지 제한
    max_tokens=512,
)

# -----------------------------------------------------------
# 실제 생성 실행
# -----------------------------------------------------------
outputs = llm.generate(
    ["Explain LLM quantization in detail."],  # 프롬프트 리스트 (여러 개를 한 번에 넣어도 됨)
    sampling_params,
)

# outputs는 프롬프트 개수만큼의 리스트이고,
# outputs[i].outputs는 각 프롬프트에 대해 생성된 후보(candidate)들의 리스트입니다.
# (기본적으로 후보 1개만 생성하므로 outputs[0].outputs[0]이 첫 번째 프롬프트의 첫 번째 생성 결과)
print(outputs[0].outputs[0].text)


## 6. 결과 해석과 다음 실험 아이디어

### 6-1. "빨라졌다"는 걸 어떻게 확인하나요?

위 코드는 결과 텍스트만 출력하지만, 실제로 속도 이득을 확인하려면 아래처럼
생성에 걸린 시간과 토큰 수를 함께 측정해서, speculative decoding을 켰을 때/껐을 때
(= speculative_config 없이 target 모델만 단독 실행)의 시간을 비교해봐야 합니다.

```python
import time

start = time.time()
outputs = llm.generate(["Explain LLM quantization in detail."], sampling_params)
elapsed = time.time() - start
num_tokens = len(outputs[0].outputs[0].token_ids)

print(f"생성된 토큰 수: {num_tokens}")
print(f"총 소요 시간: {elapsed:.2f}초")
print(f"토큰당 평균 시간: {elapsed / num_tokens * 1000:.1f}ms/token")
```

vLLM은 내부적으로 draft/target이 각각 몇 개를 제안했고 몇 개가 수락됐는지도
로그나 메트릭으로 확인할 수 있습니다 (확인 방법은 버전마다 다르니 공식 문서의 "Metrics" 섹션을 참고하세요).

### 6-2. 개인 환경에서 실습하고 싶다면? (더 가벼운 모델 조합)

70B + 8B 조합은 개인 GPU(특히 무료 Colab)로는 실행이 어렵습니다. 원리를 직접 체험해보고 싶다면
vLLM 공식 문서에도 예시로 나오는, 훨씬 가벼운 모델 조합으로 바꿔서 시도해보는 것을 추천합니다.

```python
llm = LLM(
    model="Qwen/Qwen3-8B",                 # target (약 80억 파라미터)
    speculative_config={
        "model": "Qwen/Qwen3-0.6B",        # draft (약 6억 파라미터)
        "num_speculative_tokens": 5,
        "method": "draft_model",
    },
)
```

이 조합은 훨씬 가볍기 때문에 단일 GPU에서도 개념 확인용으로 시도해볼 수 있습니다.

### 6-3. 직접 바꿔보며 실험할 것들

| 바꿔볼 값 | 관찰 포인트 |
|---|---|
| `num_speculative_tokens`를 1 → 3 → 10으로 증가 | 라운드당 평균 획득 토큰 수가 어떻게 변하는지 |
| `temperature`를 0.0 → 0.7 → 1.0으로 증가 | 수락률이 대체로 낮아지는지 (샘플링이 무작위적일수록 draft와 target이 다른 토큰을 고를 확률이 커짐) |
| draft 모델을 더 작은/더 큰 모델로 교체 | draft와 target의 "실력 차이"가 수락률에 미치는 영향 |
| 프롬프트를 다양하게 (코드, 창작, 사실 질의 등) | 태스크에 따라 수락률이 달라지는지 (반복적이고 예측하기 쉬운 텍스트일수록 대체로 수락률이 높은 경향) |

### 6-4. 참고: vLLM이 지원하는 다른 speculative decoding 방식

이 노트북은 "별도의 작은 draft 모델"을 쓰는 가장 고전적인 방식(`method="draft_model"`)을 다뤘지만,
vLLM은 다른 방식들도 지원합니다.

- **n-gram 기반** (`method="ngram"`): 입력 프롬프트나 지금까지 생성한 문장에서 반복되는 패턴을 그대로 재활용해 토큰을 제안합니다. 별도 모델이 필요 없습니다.
- **EAGLE / MTP / Medusa 등**: target 모델에 작은 "예측 head"를 추가로 붙여서, 별도의 전체 모델 없이도 초안을 제안하는 방식입니다.

방식마다 필요한 설정과 적합한 상황이 다르므로, 관심 있다면 vLLM 공식 문서의 Speculative Decoding 섹션에서
각 방식별 문서를 살펴보는 것을 추천합니다.


## 정리

- LLM 생성이 느린 이유는 "계산량"보다 "가중치를 메모리에서 읽어오는 시간" 때문인 경우가 많습니다 (memory-bandwidth-bound).
- Speculative Decoding은 작고 빠른 draft 모델이 후보 토큰을 여러 개 미리 제안하고,
  크고 정확한 target 모델이 **한 번의 forward pass**로 한꺼번에 검증하는 방식입니다.
- accept/reject 규칙을 수학적으로 잘 설계했기 때문에, **최종 결과물의 품질은 target 모델 단독 실행과 완전히 동일**합니다 (lossless). 이는 위 3번 미니 시뮬레이션에서 직접 확인했습니다.
- vLLM에서는 `model`(target)과 `speculative_config`(draft 모델, num_speculative_tokens 등)를 지정해서 바로 사용할 수 있습니다.
- 실제 속도 이득은 draft/target의 "실력 차이", 태스크의 예측 용이성, K값 등에 따라 달라지므로 직접 실험하며 감을 잡는 것이 중요합니다.
